# Pork rasher packaging inspection: full pipeline
Detect packaging defects with YOLO, then decide tray PASS/FAIL.
Use a **GPU runtime** (Runtime > Change runtime type > T4 GPU) and add a Colab secret named `ROBOFLOW_API_KEY`
(key icon in the left sidebar, with Notebook access switched on).

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/ElijahKhoury1/pork-rasher-qc.git"
REPO_DIR = "/content/pork-rasher-qc"

# Clone the first time; on later runs just pull the latest code (keeps datasets/ and runs/)
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
print(r.stdout, r.stderr)

os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")      # lets `import rasher_qc` work without pip install -e
print(sorted(os.listdir("src/rasher_qc")))

# Install dependencies (python -m pip avoids needing a shell)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ultralytics", "roboflow", "pandas", "matplotlib"],
                   capture_output=True, text=True)
print("pip exit code:", r.returncode)

In [ ]:
import torch, sys
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())
SEED = 0

## 2. Download the dataset (Version 4, YOLO format)

In [ ]:
from google.colab import userdata
from rasher_qc import data

DATASET = data.download_roboflow(userdata.get("ROBOFLOW_API_KEY"), version=4, location="datasets/pork-rasher")
DATA_YAML = str(DATASET / "data.yaml")
print(DATASET)

## 3. Check the data and remove leakage
Augmented copies of one frame must not sit in more than one split, or test scores are inflated.

In [ ]:
import pandas as pd
print(data.validate_dataset(DATASET))
counts = pd.DataFrame(data.count_classes(DATASET)).fillna(0).astype(int)
display(counts)
print("Frames appearing in more than one split:", len(data.find_cross_split_leakage(DATASET)))

In [ ]:
# Keep each leaked frame in the highest-priority split (test > valid > train); copies go to a backup folder.
try:
    from rasher_qc.leakage import remove_cross_split_leakage
except ImportError:                      # repo copy without leakage.py: use an identical built-in version
    import shutil
    from pathlib import Path
    def remove_cross_split_leakage(dataset_dir, backup_dir="datasets/removed_leaked", priority=("test", "valid", "train")):
        backup, moved = Path(backup_dir), 0
        for base, splits in data.find_cross_split_leakage(dataset_dir).items():
            keep = next(s for s in priority if s in splits)
            for split in splits - {keep}:
                for img in data.list_images(dataset_dir, split):
                    if data.base_name(img.name) != base:
                        continue
                    for f in (img, data.label_path_for(img)):
                        if f.exists():
                            dest = backup / split / f.parent.name
                            dest.mkdir(parents=True, exist_ok=True)
                            shutil.move(str(f), dest / f.name)
                    moved += 1
        return moved

print("images moved:", remove_cross_split_leakage(DATASET))
print("remaining leaks:", len(data.find_cross_split_leakage(DATASET)))

In [ ]:
# Tray-level labels per split (FAIL = at least one annotated defect)
from rasher_qc.decision import tray_label_from_yolo_txt
names = data.class_names(DATASET)
rows = []
for split in ["train", "valid", "test"]:
    labels = [tray_label_from_yolo_txt(data.label_path_for(p), names) for p in data.list_images(DATASET, split)]
    rows.append({"split": split, "images": len(labels), "FAIL": labels.count("FAIL"), "PASS": labels.count("PASS")})
display(pd.DataFrame(rows).set_index("split"))
display(pd.DataFrame(data.count_classes(DATASET)).fillna(0).astype(int))

In [ ]:
# Look at a few images
import random, matplotlib.pyplot as plt, matplotlib.image as mpimg
random.seed(SEED)
files = random.sample(data.list_images(DATASET, "train"), 6)
fig, axs = plt.subplots(2, 3, figsize=(12, 6))
for ax, f in zip(axs.ravel(), files):
    ax.imshow(mpimg.imread(f)); ax.set_title(f.name[:28], fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Train: baseline and alternatives
Baseline = `yolov8n` at 640 px. Alternatives vary model size and input resolution.
Selection uses the **validation** split only. Early stopping (`patience`) ends a run when validation stops improving.

In [ ]:
from ultralytics import YOLO
from rasher_qc import train, evaluate

EXPERIMENTS = {
    "baseline_n_640": dict(weights="yolov8n.pt", imgsz=640),
    "s_640":          dict(weights="yolov8s.pt", imgsz=640),
    "n_416":          dict(weights="yolov8n.pt", imgsz=416),
}
EPOCHS = 25      # use 3 for a quick smoke test

for name, cfg in EXPERIMENTS.items():
    train.train(DATA_YAML, epochs=EPOCHS, seed=SEED, name=name, patience=8, **cfg)
    print("finished", name)

In [ ]:
# Find where Ultralytics saved each run's best weights (the folder differs between versions)
import glob
found = glob.glob("/content/**/best.pt", recursive=True)
best = {}
for name in EXPERIMENTS:
    matches = [f for f in found if f"/{name}/weights/best.pt" in f]
    if matches:
        best[name] = matches[0]
    else:
        print("WARNING: no weights found for", name)
print(best)

## 5. Compare on the validation split (model selection)

In [ ]:
import time
import numpy as np
from ultralytics.utils.torch_utils import get_flops

def benchmark_latency(weights, images, imgsz=640, warmup=5, runs=50):
    """Latency, parameters and GFLOPs (built into the notebook; counts parameters directly)."""
    model = YOLO(str(weights))
    imgs = [str(i) for i in images]
    for i in range(warmup):
        model.predict(imgs[i % len(imgs)], imgsz=imgsz, verbose=False)
    times = []
    for i in range(runs):
        t0 = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=imgsz, verbose=False)
        times.append((time.perf_counter() - t0) * 1000)
    params = sum(p.numel() for p in model.model.parameters())
    try:
        gflops = float(get_flops(model.model, imgsz))
    except Exception:
        gflops = float("nan")
    return {"mean_ms": float(np.mean(times)), "p95_ms": float(np.percentile(times, 95)),
            "fps": float(1000 / np.mean(times)), "params_m": params / 1e6, "gflops": gflops}

rows = []
for name, cfg in EXPERIMENTS.items():
    if name not in best:
        print("skipping", name); continue
    m = evaluate.detection_metrics(best[name], DATA_YAML, split="val", imgsz=cfg["imgsz"])
    imgs = data.list_images(DATASET, "valid")[:20]
    b = benchmark_latency(best[name], imgs, imgsz=cfg["imgsz"])
    rows.append({"model": name, "mAP50": m["map50"], "mAP50-95": m["map50_95"],
                 "precision": m["precision"], "recall": m["recall"],
                 "latency_ms": b["mean_ms"], "fps": b["fps"], "params_M": b["params_m"], "GFLOPs": b["gflops"]})
results = pd.DataFrame(rows).set_index("model").round(3)
display(results)
results.to_csv("experiment_results.csv")
SELECTED = results["mAP50-95"].idxmax()
print("Selected on validation mAP50-95:", SELECTED)

## 6. Final test-set evaluation (selected model only, run once)
Detection metrics, per-class AP, and the tray-level PASS/FAIL result. The test set is small and skewed
towards FAIL trays, so also report specificity and the always-FAIL baseline.

In [ ]:
cfg = EXPERIMENTS[SELECTED]
final = evaluate.detection_metrics(best[SELECTED], DATA_YAML, split="test", imgsz=cfg["imgsz"])
print({k: round(v, 3) for k, v in final.items() if k != "per_class_map50_95"})
display(pd.Series(final["per_class_map50_95"], name="AP50-95").round(3).to_frame())

# Tray-level decision. Change FAIL_CLASSES if some classes should not cause a reject (None = all).
FAIL_CLASSES = None
tray = evaluate.tray_level_metrics(best[SELECTED], DATASET, split="test", conf=0.25,
                                   imgsz=cfg["imgsz"], fail_classes=FAIL_CLASSES)
print(tray)

n_pass = tray["tn"] + tray["fp"]
n_fail = tray["tp"] + tray["fn"]
spec = tray["tn"] / n_pass if n_pass else float("nan")
print(f"Good trays correctly passed (specificity): {spec:.3f} ({tray['tn']}/{n_pass})")
print(f"Defective trays caught (recall): {tray['recall']:.3f} ({tray['tp']}/{n_fail})")
print(f"Always-FAIL baseline: accuracy {n_fail/(n_fail+n_pass):.3f}, specificity 0.000")

## 6b. Performance profile
Per-stage timing, memory, FLOPs and a rough arithmetic-intensity estimate for each trained model (batch size 1, on the
GPU of this runtime), then an operator-level profile of the selected model.

In [ ]:
import numpy as np, torch
from ultralytics.utils.torch_utils import get_flops

PEAK_TFLOPS = 8.1   # Tesla T4 FP32 peak per NVIDIA's spec; change this if your GPU differs

rows = []
for name, cfg_ in EXPERIMENTS.items():
    if name not in best:
        continue
    imgsz = cfg_["imgsz"]
    model = YOLO(str(best[name]))
    imgs = [str(p) for p in data.list_images(DATASET, "test")[:50]]
    for p in imgs[:5]:
        model.predict(p, imgsz=imgsz, verbose=False)          # warm-up, not timed
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    pre, inf, post = [], [], []
    for p in imgs:
        s = model.predict(p, imgsz=imgsz, verbose=False)[0].speed   # ms per stage
        pre.append(s["preprocess"]); inf.append(s["inference"]); post.append(s["postprocess"])
    peak_mb = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else float("nan")

    params = sum(q.numel() for q in model.model.parameters())
    gflops = float(get_flops(model.model, imgsz))                  # GFLOPs per image
    weight_bytes = params * 4                                      # float32 weights
    inf_ms = float(np.mean(inf))
    achieved_tflops = gflops / inf_ms                              # GFLOP / ms = TFLOP/s
    rows.append({
        "model": name, "preprocess_ms": np.mean(pre), "inference_ms": inf_ms, "postprocess_ms": np.mean(post),
        "total_ms": np.mean(pre) + inf_ms + np.mean(post),
        "GFLOPs/img": gflops, "params_M": params / 1e6, "weights_MB": weight_bytes / 1e6,
        "peak_GPU_MB": peak_mb,
        "FLOP/byte (weights only)": gflops * 1e9 / weight_bytes,
        "achieved_TFLOPs": achieved_tflops,
        "pct_of_peak": 100 * achieved_tflops / PEAK_TFLOPS,
    })
perf = pd.DataFrame(rows).set_index("model").round(3)
display(perf)
perf.to_csv("performance_profile.csv")

In [ ]:
from torch.profiler import profile, ProfilerActivity

model = YOLO(str(best[SELECTED]))
img = str(data.list_images(DATASET, "test")[0])
model.predict(img, imgsz=EXPERIMENTS[SELECTED]["imgsz"], verbose=False)     # warm-up
acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if torch.cuda.is_available() else [])
with profile(activities=acts) as prof:
    model.predict(img, imgsz=EXPERIMENTS[SELECTED]["imgsz"], verbose=False)

for key in ["self_device_time_total", "self_cuda_time_total", "self_cpu_time_total"]:
    try:
        print(prof.key_averages().table(sort_by=key, row_limit=8)); break
    except Exception:
        continue

## 7. Sensitivity: confidence threshold
The threshold trades missed defects (recall) against false rejects (precision).

In [ ]:
sens = []
for c in [0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6]:
    t = evaluate.tray_level_metrics(best[SELECTED], DATASET, split="test", conf=c,
                                    imgsz=cfg["imgsz"], fail_classes=FAIL_CLASSES)
    sens.append({"conf": c, "precision": t["precision"], "recall": t["recall"], "f1": t["f1"]})
sens = pd.DataFrame(sens).set_index("conf")
display(sens.round(3))
sens.plot(marker="o", title="Tray-level metrics vs confidence threshold"); plt.grid(alpha=.3); plt.show()

## 8. Inspect individual trays and look at failures

In [ ]:
from rasher_qc.infer import inspect_tray
model = YOLO(str(best[SELECTED]))
for img in data.list_images(DATASET, "test")[:5]:
    r = inspect_tray(model, img, conf=0.25, imgsz=cfg["imgsz"], fail_classes=FAIL_CLASSES)
    print(img.name[:30], r["status"], r["counts"])

In [ ]:
# Ultralytics saves a confusion matrix and PR curves on validation runs
from IPython.display import Image, display
val_dirs = sorted(glob.glob("runs/**/val*", recursive=True), key=os.path.getmtime)
print(val_dirs[-1] if val_dirs else "no val folder found")
for f in ["confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    p = os.path.join(val_dirs[-1], f) if val_dirs else ""
    if os.path.exists(p): display(Image(p, width=600))

## 9. Generate the code docs and test report (commit these to the repo)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pdoc", "pytest", "pytest-html"])
env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src")
r = subprocess.run([sys.executable, "-m", "pdoc", "-o", "docs", "rasher_qc"], capture_output=True, text=True, env=env)
print("pdoc exit code:", r.returncode, r.stderr[-500:])
r = subprocess.run([sys.executable, "-m", "pytest", "-v", "--html=test_report.html", "--self-contained-html"],
                   capture_output=True, text=True, env=env)
print(r.stdout[-1500:])

## 10. Next steps for the report
- Look at false positives and misses on the test split: reflections, occlusion, overlapping meat.
- Download `experiment_results.csv`, `docs/`, `test_report.html` and key figures, and commit them to the repo.
- Note the limitations: tiny test classes, no `wrinkle` in test, 12 PASS trays, one camera setup.